#### **Pre-processing**

The purpose of this notebook is to build the preprocessing.py file, with which we'll preprocess the dataset to ensure reproducible results. The goal is to prepare the AdversarialQA dataset for the project. I.e. building the train/val/test splits, formating the examples into the input of the flan-t5 model (question: "..."  context: "..."), the targeted output, and tokenizing. The splits will use a SEED fixed in the config.py.

First we will pre-process the dataset, with the use of the config.py file, and after that, we will see how we built all the different functions in the pre-processing file. In order to run this notebook, you need to have cloned the complet repo

#### Load the processed dataset

In [10]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.config import PROJECT_ROOT, DATA_PROCESSED

print(PROJECT_ROOT)
print(DATA_PROCESSED.exists())

/Users/julianlilas/Desktop/DSTI/deep_learning/project/repo/DeepLearning
True


Set-up of the different parameters and config

In [3]:
# make src/ importable from the notebooks/ folder
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [4]:
from src.config import DATA_PROCESSED, MODEL_BASE, PROMPT_TEMPLATE
from src.preprocessing import load_processed
from transformers import AutoTokenizer

ds = load_processed()
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 27000
    })
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
})


Check that the imported dataset has the right processed items (input_text that include question and context, and labels that is the target output tokenized)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)

ex = ds["train"][0]
print("INPUT :", ex["input_text"][:300])
print("\nTARGET:", ex["target_text"])
print("LABELS:", tokenizer.decode(ex["labels"], skip_special_tokens=True))
print("\nlens  :", len(ex["input_ids"]), "|", len(ex["labels"]))

INPUT : question: What system is comparable to the xbox 360?  context: Virtually all console gaming systems of the previous generation used microprocessors developed by IBM. The Xbox 360 contains a PowerPC tri-core processor, which was designed and produced by IBM in less than 24 months. Sony's PlayStation 

TARGET: Nintendo's new Wii U system
LABELS: Nintendo's new Wii U system

lens  : 172 | 9


#### Below are the initial development cells, used to test and make the preprocessing.py file

In [6]:
from pathlib import Path
from datasets import (load_dataset, DatasetDict)

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

# Dataset
DATASET_NAME = "UCLNLP/adversarial_qa"
DATASET_CONFIG = "adversarialQA"

# Models (same tokenizer for both)
MODEL_BASE = "google/flan-t5-base"
MODEL_LARGE = "google/flan-t5-large"

# Prompt / length budget (from EDA)
PROMPT_TEMPLATE = "question: {question}  context: {context}"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 32
MAX_GEN_TOKENS = 48

# Reproducibility
SEED = 42
VAL_SIZE = 3000

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

PROJECT_ROOT: /Users/julianlilas/Desktop/DSTI/deep_learning/project/repo/DeepLearning
Exists: True


Split the dataset

In [ ]:
raw = load_dataset(DATASET_NAME, DATASET_CONFIG, cache_dir=DATA_RAW)

print(raw)

split = raw["train"].train_test_split(test_size=VAL_SIZE, seed=SEED)

ds = DatasetDict({
    "train": split["train"],
    "val":   split["test"],
    "test":  raw["validation"],
})

for name, dset in ds.items():
    n_empty = sum(1 for a in dset["answers"] if len(a["text"]) == 0)
    print(f"{name:6s} {len(dset):6d} rows | {n_empty:4d} without answer")

We need to define the template for the input of the model when we'll fine-tune the model

In [ ]:
def format_example(example):
    return {
        "input_text": PROMPT_TEMPLATE.format(
            question=example["question"],
            context=example["context"],
        ),
        "target_text": example["answers"]["text"][0],
    }

ds = ds.map(format_example)

ex = ds["train"][0]
print("INPUT:", ex["input_text"], sep="\n")
print("TARGET:", ex["target_text"], sep="\n")

We need now to define the tokenize function, and add all the tokenized inputs and outputs to the dataset

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)

def tokenize_example(batch):
    model_inputs = tokenizer(batch["input_text"], max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch["target_text"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

ds = ds.map(tokenize_example)

ex = ds["train"][0]
print("TARGET TEXT :", ex["target_text"])
print("LABELS DECODED:", tokenizer.decode(ex["labels"], skip_special_tokens=True))
print("INPUT LEN:", len(ex["input_ids"]), "| LABEL LEN:", len(ex["labels"]))

Finally, we need to save the different batchs (train, val, test)

In [ ]:
ds.save_to_disk(DATA_PROCESSED)

from datasets import load_from_disk
reloaded = load_from_disk(DATA_PROCESSED)
print(reloaded)